In [1]:
!pip install torchcde sktime signatory seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 46.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.6 MB/s eta 0:00:00
  Created wheel for signatory: filename=signatory-1.2.6.1.9.0-cp312-cp312-linux_x86_64.whl size=13020578 sha256=d51d7d107bf6614ec1cb42181f17ba20bdd253dead75eabbe3b0969c2009c2d1
  Stored in directory: /root/.cache/pip/wheels/ea/18/f4/25aee915ecc6b29d1a54962265ca510a889b7ea71905f446b5
Successfully built signatory


In [ ]:
# ==========================================
# 全量训练、大样本评估与对冲回测 
# ==========================================
import sys
import importlib
import torch
import torchsde
import matplotlib.pyplot as plt

# 确保能读到刚才生成的 .py 文件
sys.path.append('/kaggle/working')

# 导入所有核心模块
import data_pipeline
import generator_sde
import generator_hawkes
import eval_path_mmd
import eval_microstructure
import deep_hedging_env

# 强制重载
importlib.reload(data_pipeline)
importlib.reload(generator_sde)
importlib.reload(generator_hawkes)
importlib.reload(eval_path_mmd)
importlib.reload(eval_microstructure)
importlib.reload(deep_hedging_env)

from data_pipeline import get_universal_dataloader
from generator_sde import SDEGenerator
from generator_hawkes import MicrostructureHawkes
from eval_path_mmd import SignatureMMDLoss
from eval_microstructure import MicrostructureEvaluator
from deep_hedging_env import MicrostructureHedgingEnv, plot_hedging_results

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的计算设备: {device}")

# 1. 配置路径
TRAIN_DATA_PATH = '/kaggle/input/datasets/baoxuyang/fi2010-pt/fi2010_train.pt'

# ==========================================================
# 1：分离训练池(小批量)与评估池(大批量)
# ==========================================================
print("\n[1/6] 正在构建数据管道 (分离 Train 与 Eval)...")
train_batch_size = 128  # 用于 SGD 梯度下降，小步快跑
eval_batch_size = 2000  # 用于最终评估，大样本保证统计显著性

train_loader = get_universal_dataloader(TRAIN_DATA_PATH, batch_size=train_batch_size, shuffle=True)
eval_loader = get_universal_dataloader(TRAIN_DATA_PATH, batch_size=eval_batch_size, shuffle=True)

# 临时抓取一个以获取维度信息
_temp_batch = next(iter(train_loader))
state_dim = 41
sig_dim = _temp_batch['signature_context'].shape[-1]
future_steps = _temp_batch['clean_path'].shape[1] - 1
dt = 0.01

# 2. 初始化模型
print(f"\n[2/6] 初始化 Neural-SDE 与 Neural-Hawkes 模型...")
sde_gen = SDEGenerator(state_size=state_dim, sig_channels=sig_dim).to(device)
hawkes_gen = MicrostructureHawkes(state_size=state_dim, sig_channels=sig_dim).to(device)

# 3. 完整训练循环 (遍历所有 33,805 条数据)
print("\n[3/6] 开始在全量数据上联合训练模型 (预热)...")
optimizer_sde = torch.optim.Adam(sde_gen.parameters(), lr=1e-3)
optimizer_hawkes = torch.optim.Adam(hawkes_gen.parameters(), lr=1e-3)

sde_gen.train()
hawkes_gen.train()

train_epochs = 20 # 演示用 5 轮，Kaggle 上跑完大概几分钟。追求极致可以设为 20
for epoch in range(train_epochs):
    total_sde_loss, total_hawkes_loss = 0.0, 0.0
    for batch in train_loader:
        b_sig = batch['signature_context'].to(device)
        b_clean = batch['clean_path'].to(device)
        b_mask, b_size = batch['jump_mask'].to(device), batch['jump_size'].to(device)
        
        # Train Hawkes
        optimizer_hawkes.zero_grad()
        intensities, marks = hawkes_gen(b_mask, b_size, b_sig)
        loss_h, _, _ = hawkes_gen.compute_loss(b_mask, b_size, intensities, marks, dt=dt)
        loss_h.backward()
        optimizer_hawkes.step()
        total_hawkes_loss += loss_h.item()
        
        # Train SDE (单步预测预热)
        optimizer_sde.zero_grad()
        y0, target_y1 = b_clean[:, 0, :], b_clean[:, 1, :]
        sde_gen.sde.current_sig_context = b_sig
        ts_step = torch.tensor([0.0, dt], device=device)
        pred_y1 = torchsde.sdeint_adjoint(sde=sde_gen.sde, y0=y0, ts=ts_step, method='euler', options=dict(step_size=dt))[-1]
        loss_sde = torch.nn.functional.mse_loss(pred_y1, target_y1)
        loss_sde.backward()
        optimizer_sde.step()
        total_sde_loss += loss_sde.item()
        
    print(f"Epoch {epoch+1}/{train_epochs} | SDE MSE: {total_sde_loss/len(train_loader):.4f} | Hawkes NLL: {total_hawkes_loss/len(train_loader):.4f}")

# ==========================================================
# 2：在 2000 个大样本上进行评估与生成
# ==========================================================
print("\n[4/6] 开始在大样本 (2000条路径) 上生成对比预测...")
sde_gen.eval()
hawkes_gen.eval()

# 从 Eval Loader 抓取大批量数据
eval_batch = next(iter(eval_loader))
eval_sig = eval_batch['signature_context'].to(device)
eval_last_s = eval_batch['clean_path'][:, -1, :].to(device)
eval_real_full = eval_batch['clean_path'].to(device)
eval_curr_m = eval_batch['jump_mask'][:, -1, :].to(device)
eval_curr_sz = eval_batch['jump_size'][:, -1, :].to(device)

with torch.no_grad():
    # 模型 A: 纯 SDE (+ 均值防爆稳定器)
    fake_paths_sde = sde_gen(eval_sig, eval_last_s, future_steps=future_steps+1, dt=dt)
    fake_paths_sde = torch.clamp(fake_paths_sde, min=-10.0, max=10.0)
    
    # 模型 B: SDE-Hawkes 混合
    fake_paths_hybrid_list = [eval_last_s]
    curr_s, curr_m, curr_sz = eval_last_s, eval_curr_m, eval_curr_sz
    
    for t in range(future_steps):
        intensities, marked_jumps = hawkes_gen(curr_m.unsqueeze(1), curr_sz.unsqueeze(1), eval_sig)
        
        # 节流阀：抑制极其微小的跳跃概率噪声
        prob = torch.clamp(intensities[:, 0, :] * dt, 0.0, 1.0)
        prob = torch.where(prob < 0.05, torch.zeros_like(prob), prob)
        next_m = torch.bernoulli(prob).float()
        
        sde_gen.sde.current_sig_context = eval_sig
        ts_step = torch.tensor([0.0, dt], device=device)
        walk = torchsde.sdeint_adjoint(sde=sde_gen.sde, y0=curr_s, ts=ts_step, method='euler', options=dict(step_size=dt))[-1] 
        
        next_s = next_m * marked_jumps[:, 0, :] + (1 - next_m) * walk
        next_s = torch.clamp(next_s, min=-10.0, max=10.0)
        
        curr_s, curr_m, curr_sz = next_s, next_m, next_m * marked_jumps[:, 0, :]
        fake_paths_hybrid_list.append(curr_s)
        
    fake_paths_hybrid = torch.stack(fake_paths_hybrid_list, dim=1)

# 4. 多维量化评估
print("\n[5/6] 运行多维度微观结构评估 (Path-MMD & Stylized Facts)...")
mmd_evaluator = SignatureMMDLoss(sig_depth=2, kernel_type='linear').to(device)
real_eval_sliced = eval_real_full[:, :future_steps+1, :]

mmd_sde = mmd_evaluator(real_eval_sliced, fake_paths_sde).item()
mmd_hybrid = mmd_evaluator(real_eval_sliced, fake_paths_hybrid).item()

ms_evaluator = MicrostructureEvaluator(jump_k_sigma=3.0)
report_sde = ms_evaluator.run_full_evaluation(real_eval_sliced, fake_paths_sde)
report_hybrid = ms_evaluator.run_full_evaluation(real_eval_sliced, fake_paths_hybrid)

print("-" * 50)
print(f"{'评估指标 (2000样本)':<20} | {'纯 Neural-SDE':<15} | {'SDE-Hawkes 混合':<15}")
print("-" * 50)
print(f"{'Path-MMD (Sig空间)':<20} | {mmd_sde:<15.4f} | {mmd_hybrid:<15.4f}")
print(f"{'跳跃分布 W1 距离':<20} | {report_sde['Jump_Dist_W1_Distance']:<15.4f} | {report_hybrid['Jump_Dist_W1_Distance']:<15.4f}")
print(f"{'序列聚集 ACF MSE':<20} | {report_sde['Clustering_ACF_MSE']:<15.4f} | {report_hybrid['Clustering_ACF_MSE']:<15.4f}")
print(f"{'订单流强度误差':<20} | {report_sde['Intensity_Mean_Error']:<15.4f} | {report_hybrid['Intensity_Mean_Error']:<15.4f}")
print("-" * 50)

# ==========================================================
# 3：动态胜者路由 (Winner-Takes-All)
# ==========================================================
if mmd_sde < mmd_hybrid:
    print("=> 结论验证: 纯 Neural-SDE (连续漫游) 在当前 2000 条样本的分布拟合上表现更优。")
    best_paths, best_name = fake_paths_sde, "纯 Neural-SDE"
else:
    print("=> 结论验证: 混合模型 (离散跳跃) 在当前 2000 条样本的分布拟合上表现更优。")
    best_paths, best_name = fake_paths_hybrid, "SDE-Hawkes 混合"

# 5. 将最优路径送入 Deep Hedging 进行业务落地回测
print(f"\n[6/6] 提取特征，启动 Deep Hedging 回测 (自动选择: {best_name})...")

def extract_robust_features(paths: torch.Tensor):
    ask_p, ask_v = paths[:, :, 0], paths[:, :, 10]
    bid_p, bid_v = paths[:, :, 20], paths[:, :, 30]
    mid_prices = (ask_p + bid_p) / 2.0
    spreads = torch.abs(ask_p - bid_p) + 1e-4 
    volumes = ask_v + bid_v + 1e-4
    mid_prices = torch.clamp(torch.nan_to_num(mid_prices, nan=1.0), min=1e-3)
    spreads = torch.clamp(torch.nan_to_num(spreads, nan=0.01), min=1e-5, max=100.0)
    volumes = torch.clamp(torch.nan_to_num(volumes, nan=10.0), min=1e-2)
    return mid_prices, spreads, volumes

mid_p, spr, vol = extract_robust_features(best_paths)
mock_data_dict = {'prices': mid_p, 'spreads': spr, 'volumes': vol}

当前使用的计算设备: cuda

[1/6] 正在构建数据管道 (分离 Train 与 Eval)...
初始化数据管道：总样本数 33805，使用 3.0-Sigma 动态跳跃检验...
初始化数据管道：总样本数 33805，使用 3.0-Sigma 动态跳跃检验...

[2/6] 初始化 Neural-SDE 与 Neural-Hawkes 模型...

[3/6] 开始在全量数据上联合训练模型 (预热)...
Epoch 1/20 | SDE MSE: 0.1167 | Hawkes NLL: 0.3015
Epoch 2/20 | SDE MSE: 0.1107 | Hawkes NLL: 0.2490
Epoch 3/20 | SDE MSE: 0.1069 | Hawkes NLL: 0.2330
Epoch 4/20 | SDE MSE: 0.1043 | Hawkes NLL: 0.2237
Epoch 5/20 | SDE MSE: 0.1018 | Hawkes NLL: 0.2174
Epoch 6/20 | SDE MSE: 0.0998 | Hawkes NLL: 0.2131
Epoch 7/20 | SDE MSE: 0.0982 | Hawkes NLL: 0.2099
Epoch 8/20 | SDE MSE: 0.0963 | Hawkes NLL: 0.2071
Epoch 9/20 | SDE MSE: 0.0953 | Hawkes NLL: 0.2049
Epoch 10/20 | SDE MSE: 0.0938 | Hawkes NLL: 0.2034
Epoch 11/20 | SDE MSE: 0.0924 | Hawkes NLL: 0.2020
Epoch 12/20 | SDE MSE: 0.0912 | Hawkes NLL: 0.2005


In [ ]:
env = MicrostructureHedgingEnv(dt=dt, strike=mid_p[:, 0].mean().item(), cost_lambda=0.05)
pnl_deep, pnl_bs, act_deep, act_bs, final_prices = env.train_and_evaluate(
    train_data=mock_data_dict, test_data=mock_data_dict, epochs=300
)